In [1]:
import pandas as pd, numpy as np
from pathlib import Path
from datetime import datetime
from unidecode import unidecode
import sys, os

src = Path.cwd().parent/"src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))
pd.set_option('display.max_columns', None)

In [2]:
from routes.Paths import(
    Ruta_masterskill, 
    Ruta_opl,
    Ruta_busquedad_contacto,
    Ruta_festivos,
    Ruta_crm_ventas,
    Ruta_global, Ruta_local,
    Ruta_historico_arbol
)
from utils.Processing_OPL import data_opl
from utils.Processing_MasterSkill import limpiar_masterskill
from utils.functions import load_files_from_folder
from utils.Data_CRM.a_carga_data import load_files
from utils.Data_CRM.b_limpieza_columnas import _limpiar_columnas_crm, _clasificar_canal, _normalizar_fechas_numeros
from utils.Data_ModeloComercial.a_processing_arbol_tip_historico import _load_arbol_historico, guardar_arbol_historico

In [3]:
require_columns_contacts = {
    "id de contacto",
    "canal",
    "marca de tiempo de inicio",
    "cola",
    "agente",
    "marca de tiempo de desconexion",
    "marca de tiempo de conexion a agente",
    "marca de tiempo en cola",
    "marca de tiempo de inicio de acw",
    "marca de tiempo de fin de acw"
}
delete_dup_columns_contacts = ["id de contacto"]

df_contactos = load_files_from_folder(
    Ruta_busquedad_contacto,
    require_columns_contacts,
    delete_dup_columns_contacts
)
df_contactos = df_contactos[~df_contactos["cola"].isna()]

In [4]:
df_opl = data_opl(pd.read_excel(Ruta_opl, sheet_name="Combinado"))
df_master = limpiar_masterskill(pd.read_excel(Ruta_masterskill, sheet_name="TB"))

df_festivos = (
    pd.to_datetime(
        pd.read_csv(Ruta_festivos)["Fecha"],
        format="%d/%m/%Y"
    )
    .drop_duplicates()
    .values.astype("datetime64[D]")
)
today = pd.Timestamp.today().normalize()

df_arbol_hist = _load_arbol_historico(Ruta_historico_arbol)

In [5]:
df_crm_ventas = _normalizar_fechas_numeros(
    _clasificar_canal(
        _limpiar_columnas_crm(
            load_files(Ruta_crm_ventas)
        )
    )
)
df_crm_ventas["ID_LLAMADA_DE_CONTACTO"] = df_crm_ventas["ID_LLAMADA_DE_CONTACTO"].str.lower()
df_crm_ventas = df_crm_ventas.merge(
    df_contactos[["id de contacto", "agente", "cola"]],
    how="left",
    left_on=["ID_LLAMADA_DE_CONTACTO"],
    right_on=["id de contacto"] 
).drop(columns=["id de contacto"])

df_crm_ventas["START_MONTH_DATE"] = df_crm_ventas["FECHA_DE_INICIO"].dt.to_period("M").dt.to_timestamp()

In [6]:
guardar_arbol_historico(df_arbol_hist)